## Import Modules

In [1]:
import torch
import torch.nn as nn
import pandas as pd
import time

from torchaudio.functional import add_noise

In [2]:
BATCH_SIZE = 2048
EPOCHS = 1000
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-5
GRADIENT_CLIPPING = False
CLIP_VALUE = 100
STD_DEV_MEASURE_MOD = 50
STD_DEV_MEASURE_SCALE = 1
LOSS_REDUCTION = "mean"    # sum or mean
TRAIN_RATIO = 0.8
BETA_CYCLES=10

In [3]:
from data_processing import PatchProcessor
import data_processing

df = pd.read_csv("E:\\Coding\\vae-main\\dx7\\dx7.patchparams")
columns_to_drop = []

params = data_processing.get_params()
params = [x for x in params if x.name in df.columns]
processor = PatchProcessor(params)

df = processor.normalize(df)
df_encoded, expanded_types = processor.one_hot_dataframe(df)
mse_mask, be_mask, ce_mask, alg_mask = processor.make_masks(expanded_types)
masks = (mse_mask, be_mask, ce_mask, alg_mask)
algorithms, alibi_distances = data_processing.get_algorithms()

encoded_data_array = df_encoded.to_numpy()
x_data = torch.tensor(encoded_data_array, dtype=torch.float32)
dataset = torch.utils.data.TensorDataset(x_data)

In [4]:
from helpers import *

BETA = 0.01
mse_loss = nn.L1Loss()

def latent_display(model, name):
    train_dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    visualize_latent_space(model, dataloader=train_dataloader, device="cuda", NAME=name)

def model_size(model):
    model_parameters = filter(lambda p: p.requires_grad, model.parameters())
    params = sum([np.prod(p.size()) for p in model_parameters])
    return params

def warmup(epoch, max_epoch, rate, max_beta):
    y = min(rate * epoch, rate * max_epoch)
    y = min(y, max_beta)
    return y

def cyclic_beta_schedule(epoch, max_beta, period=100):
    # see: https://www.microsoft.com/en-us/research/blog/less-pain-more-gain-a-simple-method-for-vae-training-with-less-of-that-kl-vanishing-agony/
    half_period = period // 2
    i = epoch % period
     # hold at max for last half of period
    if i > half_period: return max_beta
     # grow to max for first half of period
    return i / half_period * max_beta

def train_model(vae, masks, train_dataloader, val_dataloader, device, debug=False, chain=False, mse_coeff=1, alg_coeff=1):
    mse_mask, be_mask, ce_mask, alg_mask = masks

    # register hook for gradient clipping
    if GRADIENT_CLIPPING:
        for param in vae.parameters():
            param.register_hook(lambda grad: torch.clamp(grad, -CLIP_VALUE, CLIP_VALUE))

    # use AdamW instead as Adam is broken
    optimizer = torch.optim.AdamW(vae.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=LEARNING_RATE,
        pct_start=0.0,
        steps_per_epoch=len(train_dataloader),
        epochs=EPOCHS
    )

    train_losses, train_mse_recon_losses, train_ce_recon_losses, train_kl_losses, train_be_recon_losses, train_algo_recon_losses  = [], [], [], [], [], []
    val_losses, val_mse_recon_losses, val_ce_recon_losses, val_kl_losses, val_be_recon_losses, val_algo_recon_losses = [], [], [], [], [], []

    alg_offset_start = 100
    alg_warmup = 100

    for epoch in range(EPOCHS):
        start_time = time.time()
        train_epoch_total_loss = 0
        train_epoch_mse_recon_loss = 0
        train_epoch_ce_recon_loss = 0
        train_epoch_be_recon_loss = 0
        train_epoch_kl_loss = 0
        train_sparsity_loss = 0
        train_epoch_algo_loss = 0
        train_total_mse_loss = 0

        val_epoch_total_loss = 0
        val_epoch_mse_recon_loss = 0
        val_epoch_ce_recon_loss = 0
        val_epoch_be_recon_loss = 0
        val_epoch_kl_loss = 0
        val_epoch_sparsity_loss = 0
        val_epoch_algo_loss = 0
        val_total_mse_loss = 0
        if epoch % STD_DEV_MEASURE_MOD == 0:
            plt.clf()

        beta = cyclic_beta_schedule(epoch, BETA, period=(EPOCHS // BETA_CYCLES))
        vae.train()
        batch_progress = 0
        for batch in train_dataloader:
            batch_progress += 1

            x_batch = batch[0].to(device)
            if epoch < alg_offset_start:
                alg_coeff_sched = 0.001
            elif epoch > alg_warmup:
                alg_coeff_sched = 1
            else:
                alg_coeff_sched = torch.sigmoid(torch.tensor(batch_progress / len(train_dataloader) / alg_warmup + (epoch - alg_offset_start) / alg_warmup))

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                recon_x, mu, logvar = vae(x_batch)

            train_total_loss, train_mse_recon_loss, train_ce_recon_loss, train_be_recon_loss, train_kl_loss, total_mse, alg_loss = vae_total_loss(
                recon_x,
                x_batch,
                be_mask,
                ce_mask,
                mse_mask,
                alg_mask,
                mu,
                logvar,
                beta,
                use_kl=True,
                mse_coeff=1,
                alg_coeff=alg_coeff_sched
            )

            #train_total_loss += drift

            optimizer.zero_grad()
            train_total_loss.backward()
            optimizer.step()
            scheduler.step()

            train_epoch_total_loss += train_total_loss.item()
            train_epoch_mse_recon_loss += train_mse_recon_loss.item()
            train_epoch_ce_recon_loss += train_ce_recon_loss.item()
            train_epoch_be_recon_loss += train_be_recon_loss.item()
            train_epoch_kl_loss += train_kl_loss.item()
            train_epoch_algo_loss += alg_loss.item()
            train_total_mse_loss += total_mse.item()

        vae.eval() # <- mode for just evaluating
        with torch.no_grad(): # <- don't track gradients
            for batch in val_dataloader:
                x_batch = batch[0].to(device)
                recon_x, mu, logvar = vae(x_batch)

                drift = torch.tensor(0.0).to(device)
                if chain:
                    vae.reparameterization = False
                    double_reconstruction, _, _, _ = vae(recon_x)
                    triple_reconstruction, _, _, _ = vae(double_reconstruction)
                    vae.reparameterization = True

                    drift += mse_loss(recon_x, double_reconstruction)
                    drift += mse_loss(double_reconstruction, triple_reconstruction)

                val_total_loss, val_mse_recon_loss, val_ce_recon_loss, val_be_recon_loss, val_kl_loss, val_mse_total, alg_loss = vae_total_loss(
                    recon_x,
                    x_batch,
                    be_mask,
                    ce_mask,
                    mse_mask,
                    alg_mask,
                    mu,
                    logvar,
                    beta,
                )

                val_total_loss += drift

                val_epoch_total_loss += val_total_loss.item()
                val_epoch_mse_recon_loss += val_mse_recon_loss.item()
                val_epoch_ce_recon_loss += val_ce_recon_loss.item()
                val_epoch_be_recon_loss += val_be_recon_loss.item()
                val_epoch_kl_loss += val_kl_loss.item()
                val_total_mse_loss += val_mse_total.item()
                val_epoch_algo_loss += alg_loss.item()

        train_avg_total_loss = train_epoch_total_loss           / len(train_dataloader)
        train_avg_mse_recon_loss = train_epoch_mse_recon_loss   / len(train_dataloader)
        train_avg_ce_recon_loss = train_epoch_ce_recon_loss     / len(train_dataloader)
        train_avg_be_recon_loss = train_epoch_be_recon_loss     / len(train_dataloader)
        train_avg_kl_loss = train_epoch_kl_loss                 / len(train_dataloader)
        train_avg_algo_loss = train_epoch_algo_loss             / len(train_dataloader)
        train_avg_epoch_mse_loss = train_total_mse_loss         / len(train_dataloader)

        train_losses.append(train_avg_total_loss)
        train_mse_recon_losses.append(train_avg_mse_recon_loss)
        train_ce_recon_losses.append(train_avg_ce_recon_loss)
        train_be_recon_losses.append(train_avg_be_recon_loss)
        train_algo_recon_losses.append(train_avg_algo_loss)

        train_kl_losses.append(train_avg_kl_loss)

        val_avg_total_loss = val_epoch_total_loss               / len(val_dataloader)
        val_avg_mse_recon_loss = val_epoch_mse_recon_loss       / len(val_dataloader)
        val_avg_ce_recon_loss = val_epoch_ce_recon_loss         / len(val_dataloader)
        val_avg_be_recon_loss = val_epoch_be_recon_loss         / len(val_dataloader)
        val_avg_kl_loss = val_epoch_kl_loss                     / len(val_dataloader)
        val_avg_algo_loss = val_epoch_algo_loss                 / len(val_dataloader)
        val_avg_epoch_mse_loss = val_total_mse_loss             / len(val_dataloader)

        val_losses.append(val_avg_total_loss)
        val_mse_recon_losses.append(val_avg_mse_recon_loss)
        val_ce_recon_losses.append(val_avg_ce_recon_loss)
        val_be_recon_losses.append(val_avg_be_recon_loss)
        val_algo_recon_losses.append(val_avg_algo_loss)
        val_kl_losses.append(val_avg_kl_loss)

        end_time = time.time()
        runtime = end_time - start_time

        if debug:
            print("-" * 50)
            print(f"Epoch {epoch+1}/{EPOCHS} done in {runtime:.4f} seconds")
            print(f"Beta: {beta:.4f}")
            print(f"Learning Rate: {scheduler.get_last_lr()[0]:.8f}")

            print(f"Training Loss: {train_avg_total_loss:.4f}")
            print(f"\tTraining L1 Reconstruction Loss: {train_avg_mse_recon_loss:.4f}")
            print(f"\tTraining CE Reconstruction Loss: {train_avg_ce_recon_loss:.4f}")
            print(f"\tTraining BE Reconstruction Loss: {train_avg_be_recon_loss:.4f}")
            print(f"\tTraining Algorithm Loss: {train_avg_algo_loss:.4f}")
            print(f"\tTraining KL Loss: {train_avg_kl_loss:.4f}")
            print(f"\tTraining Total MSE Over All Params: {train_avg_epoch_mse_loss:.4f}")

            print(f"Validation Loss: {val_avg_total_loss:.4f}")
            print(f"\tValidation L1 Reconstruction Loss: {val_avg_mse_recon_loss:.4f}")
            print(f"\tValidation CE Reconstruction Loss: {val_avg_ce_recon_loss:.4f}")
            print(f"\tValidation BE Reconstruction Loss: {val_avg_be_recon_loss:.4f}")
            print(f"\tValidation Algorithm Loss: {val_avg_algo_loss:.4f}")
            print(f"\tValidation KL Loss: {val_avg_kl_loss:.4f}")
            print(f"\tValidation Total MSE Over All Params: {val_avg_epoch_mse_loss:.4f}")

            print("-" * 50)

    return vae, train_losses, train_mse_recon_losses, train_ce_recon_losses, train_kl_losses, train_be_recon_losses, train_algo_recon_losses, val_losses, val_mse_recon_losses, val_ce_recon_losses, val_be_recon_losses, val_kl_losses, val_algo_recon_losses

def train_loop(model, chain=False, mse_coeff=1.0):
    # determine device
    has_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if has_cuda else "cpu")
    if has_cuda:
        print("Using GPU")
    else:
        print("Using CPU")

    train_data_count = round(len(dataset) * TRAIN_RATIO)
    val_data_count = len(dataset) - train_data_count

    print("Total samples:", len(dataset))
    print("Training samples:", train_data_count)
    print("Validation samples:", val_data_count)

    train_data, val_data = torch.utils.data.random_split(dataset, [train_data_count, val_data_count])
    train_dataloader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = torch.utils.data.DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=True)

    model.to(device=device)

    print(f"{model_size(model)} Parameters")

    # train vae
    print("Training VAE...")
    vae, train_losses, train_mse_recon_losses, train_ce_recon_losses, train_kl_losses, train_be_recon_losses, train_algo_recon_losses, val_losses, val_mse_recon_losses, val_ce_recon_losses, val_be_recon_losses, val_kl_losses, val_algo_recon_loss = train_model(model, masks, train_dataloader, val_dataloader, device, debug=True, chain=chain, mse_coeff=mse_coeff)
    print("VAE complete.")

    # visualize results
    print("Plotting loss and UMAP of latent space...")
    os.makedirs("./models/" + NAME, exist_ok=True)
    plot_loss(train_losses, train_mse_recon_losses, train_ce_recon_losses, train_kl_losses, train_be_recon_losses, train_algo_recon_losses, val_losses, val_mse_recon_losses, val_ce_recon_losses, val_be_recon_losses, val_algo_recon_loss, val_kl_losses, NAME)

    train_dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)
    visualize_latent_space(model, dataloader=train_dataloader, device="cuda", NAME=NAME)

    # save the vae parameters
    torch.save(vae.state_dict(), "./models/" + NAME + "/model.pth")
    torch.save(vae, "./models/" + NAME + "/model.model")

    summary_name = "./models/" + NAME + "/summary.txt"

    with open(summary_name, 'w') as f:
        f.write(f"Train Total Loss: {train_losses[-1]}\n")
        f.write(f"Train MSE Loss: {train_mse_recon_losses[-1]}\n")
        f.write(f"Train CE Loss: {train_ce_recon_losses[-1]}\n")
        f.write(f"Train BE Loss: {train_be_recon_losses[-1]}\n")
        f.write(f"Train Algorithm CE Loss: {train_algo_recon_losses[-1]}\n")

        f.write(f"Validation Total Loss: {train_losses[-1]}\n")
        f.write(f"Validation MSE Loss: {train_mse_recon_losses[-1]}\n")
        f.write(f"Validation CE Loss: {train_ce_recon_losses[-1]}\n")
        f.write(f"Validation BE Loss: {train_be_recon_losses[-1]}\n")
        f.write(f"Validation Algorithm CE Loss: {train_algo_recon_losses[-1]}\n")

    evaluate_model(model, masks, val_dataloader)

In [5]:
import math
from collections import deque
import os
import time
import torch
import torch.nn.functional as F
import tqdm

from dataset import SpectrogramLatentStreamingDataset, SpectrogramStreamingDataset

def train_diffusion_model(
    model,
    train_dataloader,
    val_dataloader,
    device,
    epochs,
    learning_rate=1e-4,
    weight_decay=1e-2,
    debug=False,
):
    """
    Train the DX7-conditioned AudioLDM2 diffusion model.

    Each batch should provide:

        batch[0] -> patch information
        batch[1] -> clean AudioLDM2 latent

    The patch is expected to either already be split into
    operators/global parameters or be handled by the Dataset.

    The model performs:

        clean latent
            |
            + noise @ timestep t
            |
            v
        noisy latent
            |
            + patch conditioning
            |
            v
        AudioLDM2 U-Net
            |
            v
        predicted noise

    Loss = MSE(predicted_noise, true_noise)
    """

    model.to(device)

    # -------------------------------------------------------------
    # Optimizer
    # -------------------------------------------------------------

    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=learning_rate,
        weight_decay=weight_decay,
    )

    # OneCycle is reasonable, but unlike the VAE training I would
    # keep the schedule simple initially.
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=learning_rate,
        pct_start=0.05,
        steps_per_epoch=len(train_dataloader),
        epochs=epochs,
    )

    # -------------------------------------------------------------
    # Loss history
    # -------------------------------------------------------------

    train_losses = []
    val_losses = []

    # -------------------------------------------------------------
    # Diffusion timestep range
    # -------------------------------------------------------------

    num_train_timesteps = (
        model.scheduler.config.num_train_timesteps
    )

    for epoch in range(epochs):
        start_time = time.time()

        # =========================================================
        # TRAIN
        # =========================================================

        model.train()

        train_epoch_loss = 0.0
        pbar = tqdm.tqdm(train_dataloader)

        moving_average_len = 250
        moving_average = deque(maxlen=moving_average_len)
        step = 0
        scaler = torch.amp.GradScaler("cuda")

        for batch in pbar:
            target_mels, patches = batch

            patches = patches.to(
                device=device,
            )

            target_mels = target_mels.to(
                device=device,
            )

            target_mels = target_mels.unsqueeze(1)
            patches = patches.squeeze(1)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
            ):
                loss = model(patches, target_mels)

            # scaler.scale(loss).backward()
            # scaler.step(optimizer)
            # scaler.update()

            # Backprop

            # Optional gradient clipping
            if GRADIENT_CLIPPING:
                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    CLIP_VALUE,
                )

            loss.backward()
            optimizer.step()
            scheduler.step()

            train_epoch_loss += loss.item()


            moving_average.append(loss.item())

            pbar.set_description(f"Epoch: {epoch} --- Moving Average Loss: {sum(moving_average) / min(max(step, 1), moving_average_len)}")

            if step % moving_average_len == 0:
                print(f"Training Loss: {sum(moving_average) / moving_average_len}")

            step += 1

        train_avg_loss = (
            train_epoch_loss / len(train_dataloader)
        )

        train_losses.append(train_avg_loss)

        # =========================================================
        # VALIDATION
        # =========================================================

        model.eval()

        val_epoch_loss = 0.0

        with torch.no_grad():

            for batch in val_dataloader:

                patch = batch[0]

                target_latents = batch[1].to(
                    device=device,
                    dtype=next(model.parameters()).dtype,
                )

                operators = patch["operators"]
                global_params = patch["global_params"]

                operators = [
                    x.to(
                        device=device,
                        dtype=target_latents.dtype,
                    )
                    for x in operators
                ]

                global_params = global_params.to(
                    device=device,
                    dtype=target_latents.dtype,
                )

                batch_size = target_latents.shape[0]

                # -------------------------------------------------
                # Random validation timestep
                # -------------------------------------------------

                timesteps = torch.randint(
                    0,
                    num_train_timesteps,
                    (batch_size,),
                    device=device,
                    dtype=torch.long,
                )

                noise = torch.randn_like(target_latents)

                noisy_latents = model.scheduler.add_noise(
                    target_latents,
                    noise,
                    timesteps,
                )

                # -------------------------------------------------
                # Condition
                # -------------------------------------------------

                encoder_hidden_states, encoder_hidden_states_1 = (
                    model.condition(
                        operators,
                        global_params,
                    )
                )

                # -------------------------------------------------
                # Predict noise
                # -------------------------------------------------

                noise_pred = model.predict_noise(
                    noisy_latents=noisy_latents,
                    timesteps=timesteps,
                    encoder_hidden_states=encoder_hidden_states,
                    encoder_hidden_states_1=encoder_hidden_states_1,
                )

                val_loss = F.mse_loss(
                    noise_pred.float(),
                    noise.float(),
                )

                val_epoch_loss += val_loss.item()

        val_avg_loss = (
            val_epoch_loss / len(val_dataloader)
        )

        val_losses.append(val_avg_loss)

        # =========================================================
        # LOGGING
        # =========================================================

        runtime = time.time() - start_time

        if debug:

            print("-" * 60)

            print(
                f"Epoch {epoch + 1}/{epochs} "
                f"done in {runtime:.2f} seconds"
            )

            print(
                f"Learning Rate: "
                f"{scheduler.get_last_lr()[0]:.8f}"
            )

            print(
                f"Training Diffusion Loss: "
                f"{train_avg_loss:.6f}"
            )

            print(
                f"Validation Diffusion Loss: "
                f"{val_avg_loss:.6f}"
            )

            print("-" * 60)

    return (
        model,
        train_losses,
        val_losses,
    )

def train_loop_diffusion(
    model,
    epochs=8,
    learning_rate=1e-4,
):

    dataset = SpectrogramStreamingDataset("E:\\SynthesizerDataset\\single_note_dataset\\compressed_specs\\", "E:\\SynthesizerDataset\\patches\\")

    # dataset = SpectrogramLatentStreamingDataset("D:\\SynthesizerDataset\\ldm2_latents\\", "E:\\SynthesizerDataset\\patches\\")

    has_cuda = torch.cuda.is_available()
    device = torch.device("cuda" if has_cuda else "cpu")
    if has_cuda:
        print("Using GPU")
    else:
        print("Using CPU")

    train_data_count = round(len(dataset) * TRAIN_RATIO)
    val_data_count = len(dataset) - train_data_count

    print("Total samples:", len(dataset))
    print("Training samples:", train_data_count)
    print("Validation samples:", val_data_count)

    train_data, val_data = torch.utils.data.random_split(dataset, [train_data_count, val_data_count])
    train_dataloader = torch.utils.data.DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_dataloader = torch.utils.data.DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=True)

    (
        model,
        train_losses,
        val_losses,
    ) = train_diffusion_model(
        model=model,
        train_dataloader=train_dataloader,
        val_dataloader=val_dataloader,
        device=device,
        epochs=epochs,
        learning_rate=learning_rate,
        debug=True,
    )

    model.to(device=device)

    print("Diffusion training complete.")

    # -------------------------------------------------------------
    # Save
    # -------------------------------------------------------------

    os.makedirs(
        "./models/" + NAME,
        exist_ok=True,
    )

    torch.save(
        model.state_dict(),
        "./models/" + NAME + "/model.pth",
    )

    torch.save(
        model,
        "./models/" + NAME + "/model.model",
    )

    # -------------------------------------------------------------
    # Summary
    # -------------------------------------------------------------

    summary_name = (
        "./models/" + NAME + "/summary.txt"
    )

    with open(summary_name, "w") as f:

        f.write(
            f"Train Diffusion Loss: "
            f"{train_losses[-1]}\n"
        )

        f.write(
            f"Validation Diffusion Loss: "
            f"{val_losses[-1]}\n"
        )

    return model, train_losses, val_losses

In [6]:
from torchmetrics.classification import BinaryAUROC, BinaryPrecisionRecallCurve
from sklearn.metrics import auc

c_lengths = [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6]

import torch
import torch.nn.functional as F
from torchmetrics.classification import (
    MulticlassPrecision,
    MulticlassRecall,
    MulticlassF1Score,
    MulticlassAUROC,
    MulticlassAveragePrecision
)

def update_grouped_categorical_metrics(
    pred,
    target,
    ce_mask,
    c_lengths,
    categorical_metrics
):

    logits_all = pred[:, ce_mask]       # [B, sum(c_lengths)]
    targets_all = target[:, ce_mask]    # [B, sum(c_lengths)]

    start = 0

    for group_idx, length in enumerate(c_lengths):
        end = start + length

        logits_group = logits_all[:, start:end]     # [B, C]
        targets_group = targets_all[:, start:end]   # [B, C]

        # one-hot → class index
        target_indices = targets_group.argmax(dim=1)

        # softmax probabilities
        probs = F.softmax(logits_group, dim=1)

        # unpack metrics for this group
        (
            precision_metric,
            recall_metric,
            f1_metric,
            auroc_metric,
            prauc_metric
        ) = categorical_metrics[group_idx]

        if probs.shape[1] != length:
            pass

        # update metrics
        precision_metric.update(probs, target_indices)
        recall_metric.update(probs, target_indices)
        f1_metric.update(probs, target_indices)
        auroc_metric.update(probs, target_indices)
        prauc_metric.update(probs, target_indices)

        start = end

def compute_grouped_categorical_metrics(categorical_metrics):
    results = []

    for metrics in categorical_metrics:
        group_values = [m.compute() for m in metrics]
        results.append(torch.stack(group_values))

    return results


def init_categorical_metrics(c_lengths, device=None):

    categorical_metrics = []

    for num_classes in c_lengths:
        precision = MulticlassPrecision(
            num_classes=num_classes,
            average="macro"
        )

        recall = MulticlassRecall(
            num_classes=num_classes,
            average="macro"
        )

        f1 = MulticlassF1Score(
            num_classes=num_classes,
            average="macro"
        )

        auroc = MulticlassAUROC(
            num_classes=num_classes
        )

        prauc = MulticlassAveragePrecision(
            num_classes=num_classes
        )

        if device is not None:
            precision = precision.to(device)
            recall = recall.to(device)
            f1 = f1.to(device)
            auroc = auroc.to(device)
            prauc = prauc.to(device)

        categorical_metrics.append(
            (precision, recall, f1, auroc, prauc)
        )

    return categorical_metrics


def evaluate(pred, target, binary_metrics, categorical_metrics, algorithm_metrics, masks):
    device = pred.device

    mse_mask, be_mask, ce_mask, alg_mask = masks

    be_mask = be_mask.to(device)
    ce_mask = ce_mask.to(device)
    mse_mask = mse_mask.to(device)
    alg_mask = alg_mask.to(device)

    if be_mask.any():
        # Slice masked logits and targets
        logits = pred[:, be_mask]        # [B, num_be_features]
        targets = target[:, be_mask]     # [B, num_be_features]

        # Convert logits → probabilities
        probs = torch.sigmoid(logits)    # [B, num_be_features]

        # Make sure targets are floats (0/1)
        targets = targets.float()        # [B, num_be_features]

        # Unpack metrics tuple
        auroc_metric, prauc_metric = binary_metrics

        # Update metrics
        auroc_metric.update(probs, targets)
        # Loop over BE features

        for i in range(probs.shape[1]):
            # shape [B]
            prob_i = probs[:, i]
            target_i = targets[:, i].long()  # must be int/long, shape [B]

            # Update per-feature PR curve
            prauc_metric.update(prob_i, target_i)

    update_grouped_categorical_metrics(pred, target, ce_mask, c_lengths, categorical_metrics)

    if alg_mask.any():
        logits = pred[:, alg_mask]
        targets = target[:, alg_mask].float()

        probs = F.softmax(logits, dim=1)
        targets = targets.argmax(dim=1)

        auroc_metric, prauc_metric = algorithm_metrics

        auroc_metric.update(probs, targets)
        prauc_metric.update(probs, targets)


def evaluate_model(vae, masks, train_dataloader):
    vae.eval() # <- mode for just evaluating

    with (torch.no_grad()): # <- don't track gradients
        be_auroc_metric = BinaryAUROC()
        be_prauc_metric = BinaryPrecisionRecallCurve()
        binary_metrics = (be_auroc_metric, be_prauc_metric)

        categorical_metrics = init_categorical_metrics(c_lengths)

        num_classes = 32
        alg_auroc = MulticlassAUROC(
                num_classes=num_classes
            )

        alg_prauc = MulticlassAveragePrecision(
            num_classes=num_classes
        )

        algorithm_metrics = (alg_auroc, alg_prauc)

        device = "cuda"
        vae = vae.to(device)
        for batch in train_dataloader:
            x_batch = batch[0].to(device)

            recon_x, mu, logvar = vae(x_batch)

            recon_x = recon_x.to("cpu")
            x_batch = x_batch.to("cpu")

            evaluate(
                recon_x,
                x_batch,
                binary_metrics,
                categorical_metrics,
                algorithm_metrics,
                masks
            )

        categorical_results = compute_grouped_categorical_metrics(categorical_metrics)

        be_auroc_metric, be_prauc_metric = binary_metrics
        binary_auroc_result = be_auroc_metric.compute()
        binary_prauc_prec, binary_prauc_recall, _ = be_prauc_metric.compute()

        ce_auroc_metric, ce_prauc_metric = algorithm_metrics
        alg_auroc_result = ce_auroc_metric.compute()
        alg_prauc_pred = ce_prauc_metric.compute()

        for index in range(len(categorical_results)):
            group = categorical_results[index]
            name = names_in_order[index]
            print("-" * 14 + " " + name + " " + "-" * 14)
            print(f"precision: {group[0]}")
            print(f"recall: {group[1]}")
            print(f"f1: {group[2]}")
            print(f"auroc: {group[3]}")
            print(f"prauc: {group[4]}\n")

        averages = torch.stack(categorical_results).mean(dim=0)

        print(f"Average Categorical Var Precision: {averages[0]}")
        print(f"Average Categorical Var recall: {averages[1]}")
        print(f"Average Categorical Var f1: {averages[2]}")
        print(f"Average Categorical Var auroc: {averages[3]}")
        print(f"Average Categorical Var prauc: {averages[4]}\n")

        print(f"Binary: AUROC-{binary_auroc_result:.04}\tPRROC-{auc(binary_prauc_recall.cpu().numpy(),
                                                                    binary_prauc_prec.cpu().numpy()):.04}")
        print(f"Algorithm: AUROC-{alg_auroc_result:.04}\tPRROC-{alg_prauc_pred:.04}")


In [7]:
names = ["operator6rate1", "operator6rate2", "operator6rate3", "operator6rate4", "operator6level1", "operator6level2", "operator6level3", "operator6level4", "operator6keyboardlevelscalingbreakpoint", "operator6keyboardlevelscalingleftdepth", "operator6keyboardlevelscalingrightdepth", "operator6keyboardlevelscalingleftcurve", "operator6keyboardlevelscalingrightcurve", "operator6keyboardratescaling", "operator6amplitudemodulationsensitivity", "operator6keyvelocitysensitivity", "operator6operatoroutputlevel", "operator6oscillatormode", "operator6frequencycoarse", "operator6frequencyfine", "operator6frequencydetune", "operator5rate1", "operator5rate2", "operator5rate3", "operator5rate4", "operator5level1", "operator5level2", "operator5level3", "operator5level4", "operator5keyboardlevelscalingbreakpoint", "operator5keyboardlevelscalingleftdepth", "operator5keyboardlevelscalingrightdepth", "operator5keyboardlevelscalingleftcurve", "operator5keyboardlevelscalingrightcurve", "operator5keyboardratescaling", "operator5amplitudemodulationsensitivity", "operator5keyvelocitysensitivity", "operator5operatoroutputlevel", "operator5oscillatormode", "operator5frequencycoarse", "operator5frequencyfine", "operator5frequencydetune", "operator4rate1", "operator4rate2", "operator4rate3", "operator4rate4", "operator4level1", "operator4level2", "operator4level3", "operator4level4", "operator4keyboardlevelscalingbreakpoint", "operator4keyboardlevelscalingleftdepth", "operator4keyboardlevelscalingrightdepth", "operator4keyboardlevelscalingleftcurve", "operator4keyboardlevelscalingrightcurve", "operator4keyboardratescaling", "operator4amplitudemodulationsensitivity", "operator4keyvelocitysensitivity", "operator4operatoroutputlevel", "operator4oscillatormode", "operator4frequencycoarse", "operator4frequencyfine", "operator4frequencydetune", "operator3rate1", "operator3rate2", "operator3rate3", "operator3rate4", "operator3level1", "operator3level2", "operator3level3", "operator3level4", "operator3keyboardlevelscalingbreakpoint", "operator3keyboardlevelscalingleftdepth", "operator3keyboardlevelscalingrightdepth", "operator3keyboardlevelscalingleftcurve", "operator3keyboardlevelscalingrightcurve", "operator3keyboardratescaling", "operator3amplitudemodulationsensitivity", "operator3keyvelocitysensitivity", "operator3operatoroutputlevel", "operator3oscillatormode", "operator3frequencycoarse", "operator3frequencyfine", "operator3frequencydetune", "operator2rate1", "operator2rate2", "operator2rate3", "operator2rate4", "operator2level1", "operator2level2", "operator2level3", "operator2level4", "operator2keyboardlevelscalingbreakpoint", "operator2keyboardlevelscalingleftdepth", "operator2keyboardlevelscalingrightdepth", "operator2keyboardlevelscalingleftcurve", "operator2keyboardlevelscalingrightcurve", "operator2keyboardratescaling", "operator2amplitudemodulationsensitivity", "operator2keyvelocitysensitivity", "operator2operatoroutputlevel", "operator2oscillatormode", "operator2frequencycoarse", "operator2frequencyfine", "operator2frequencydetune", "operator1rate1", "operator1rate2", "operator1rate3", "operator1rate4", "operator1level1", "operator1level2", "operator1level3", "operator1level4", "operator1keyboardlevelscalingbreakpoint", "operator1keyboardlevelscalingleftdepth", "operator1keyboardlevelscalingrightdepth", "operator1keyboardlevelscalingleftcurve", "operator1keyboardlevelscalingrightcurve", "operator1keyboardratescaling", "operator1amplitudemodulationsensitivity", "operator1keyvelocitysensitivity", "operator1operatoroutputlevel", "operator1oscillatormode", "operator1frequencycoarse", "operator1frequencyfine", "operator1frequencydetune", "pitchegrate1", "pitchegrate2", "pitchegrate3", "pitchegrate4", "pitcheglevel1", "pitcheglevel2", "pitcheglevel3", "pitcheglevel4", "algorithm", "feedback", "oscillatorkeysync", "lfospeed", "lfodelay", "lfopitchmodulationdepth", "lfoamplitudemodulationdepth", "lfokeysync", "lfowave", "lfopitchmodulationsensitivity", "transpose"]
types = ["m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "c", "c", "m", "m", "m", "m", "b", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "c", "c", "m", "m", "m", "m", "b", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "c", "c", "m", "m", "m", "m", "b", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "c", "c", "m", "m", "m", "m", "b", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "c", "c", "m", "m", "m", "m", "b", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "c", "c", "m", "m", "m", "m", "b", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "m", "c", "m", "b", "m", "m", "m", "m", "b", "c", "m", "m"]

names_in_order = []

for index in range(len(types)):
    if types[index] == "c" and names[index] != "algorithm":
        names_in_order.append(names[index])

In [8]:
import numpy as np

number_of_params = 225
global_params = number_of_params - 28 * 6
sizes = [28, 28, 28, 28, 28, 28, global_params]

def expand_adjacency(A, sizes):
    B_size, n, _ = A.shape
    assert len(sizes) == n

    total_size = sum(sizes)
    B = np.zeros((B_size, total_size, total_size))

    row_start = 0
    for i in range(n):
        row_end = row_start + sizes[i]

        col_start = 0
        for j in range(n):
            col_end = col_start + sizes[j]

            # Fill block for ALL batches at once
            B[:, row_start:row_end, col_start:col_end] = A[:, i, j][:, None, None]

            col_start = col_end

        row_start = row_end

    return B

In [9]:
## Ideas
# Masking and Patchout as augmentation and normalization
# Model randomness added by resampling as a different patch and a scale
# Final output is final patch minus the patch that the randomness cooresponds to

In [10]:
BETA = 5e-2
LATENT_DIM = 32
EPOCHS = 800
number_of_params = 225
BATCH_SIZE = int(256 * 1.25)
mask_type = None
TRAIN_RATIO = 1 - 1.0/20.0

In [ ]:
from GraphTransformerAlibiGlobalToken import GraphTransformerAutoencoderAlibi

global_params = number_of_params - 28 * 6 - 32
algorithm_index = 176

NAME = f"GraphTransformer-Alibi-Global-Token-ALIBI-Modulation-Latent-{LATENT_DIM}-Beta-{BETA}-Batch-{BATCH_SIZE}"
model = GraphTransformerAutoencoderAlibi(input_size=28, latent_space=LATENT_DIM, d_model=256, depth=6, heads=4, mlp_dim=512, num_algorithms=32, num_global_params=global_params, sparsity_weight=1, reparameterization=True, algorithm_distance_matricies=alibi_distances, hard_prediction=False, geom_dim=16)
train_loop(model, chain=False)

In [ ]:
from TransformerAutoencoder import TransformerAutoencoder

number_of_params = 225

expanded_adjacency = expand_adjacency(alibi_distances, sizes)

NAME = f"PlainTransformer-Latent-{LATENT_DIM}-Beta-{BETA}-Batch-{BATCH_SIZE}"
model = TransformerAutoencoder(input_size=number_of_params, latent_space=LATENT_DIM, d_model=256, depth=6, heads=4, mlp_dim=512, reparameterization=True, adjacency_matrix=expanded_adjacency)
train_loop(model, chain=False)

# GENERATION

In [11]:
from diffusers import AudioLDM2Pipeline

pipe = AudioLDM2Pipeline.from_pretrained(
    "cvssp/audioldm2-music",
    torch_dtype=torch.float32,
)

E:\Coding\SongAnalyzer\.venv-flash\Lib\site-packages\diffusers\utils\deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


Loading pipeline components...:   0%|          | 0/11 [00:00<?, ?it/s]

[transformers] You are using a model of type `hifigan` to instantiate a model of type `speecht5_hifigan`. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/555 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Expected types for language_model: (<class 'transformers.models.gpt2.modeling_gpt2.GPT2LMHeadModel'>,), got <class 'transformers.models.gpt2.modeling_gpt2.GPT2Model'>.


In [12]:
print(pipe.unet.config.cross_attention_dim)
print(pipe.unet.config)

[[None, 768, 1024], [None, 768, 1024], [None, 768, 1024], [None, 768, 1024]]
FrozenDict({'sample_size': 256, 'in_channels': 8, 'out_channels': 8, 'flip_sin_to_cos': True, 'freq_shift': 0, 'down_block_types': ['DownBlock2D', 'CrossAttnDownBlock2D', 'CrossAttnDownBlock2D', 'CrossAttnDownBlock2D'], 'mid_block_type': 'UNetMidBlock2DCrossAttn', 'up_block_types': ['CrossAttnUpBlock2D', 'CrossAttnUpBlock2D', 'CrossAttnUpBlock2D', 'UpBlock2D'], 'only_cross_attention': False, 'block_out_channels': [128, 256, 384, 640], 'layers_per_block': 2, 'downsample_padding': 1, 'mid_block_scale_factor': 1, 'act_fn': 'silu', 'norm_num_groups': 32, 'norm_eps': 1e-05, 'cross_attention_dim': [[None, 768, 1024], [None, 768, 1024], [None, 768, 1024], [None, 768, 1024]], 'transformer_layers_per_block': 1, 'attention_head_dim': 8, 'num_attention_heads': None, 'use_linear_projection': False, 'class_embed_type': None, 'num_class_embeds': None, 'upcast_attention': False, 'resnet_time_scale_shift': 'default', 'time_em

In [13]:
import inspect

print(inspect.signature(pipe.unet.forward))

(sample: torch.Tensor, timestep: torch.Tensor | float | int, encoder_hidden_states: torch.Tensor, class_labels: torch.Tensor | None = None, timestep_cond: torch.Tensor | None = None, attention_mask: torch.Tensor | None = None, cross_attention_kwargs: dict[str, typing.Any] | None = None, encoder_attention_mask: torch.Tensor | None = None, return_dict: bool = True, encoder_hidden_states_1: torch.Tensor | None = None, encoder_attention_mask_1: torch.Tensor | None = None) -> diffusers.models.unets.unet_2d_condition.UNet2DConditionOutput | tuple


In [14]:
from DiffusionGenerator import DX7AudioLDM2

model = DX7AudioLDM2(
    audioldm=pipe,

    # Example -- replace these with your actual DX7 grouping.
    number_of_params=225,

    # These need to match the actual U-Net cross-attention
    # dimensions from your checkpoint.
    stream_1_dim=768,
    stream_2_dim=1024,

    hidden_dim=512,
    num_layers=2,
    num_heads=8,
).cuda()

model.train()
params = list(model.parameters())
print(len(params))

E:\Coding\SongAnalyzer\.venv-flash\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


1950


In [15]:
del pipe

In [ ]:
BATCH_SIZE = 3

train_loop_diffusion(
    model,
    epochs=4,
    learning_rate=1e-4,
)

Using GPU
Total samples: 1003117
Training samples: 952961
Validation samples: 50156


Epoch: 0 --- Moving Average Loss: 0.32606765627861023:   0%|          | 1/317654 [00:09<813:29:08,  9.22s/it]

Training Loss: 0.0013042706251144408


Epoch: 0 --- Moving Average Loss: 0.12097258288390003:   0%|          | 251/317654 [03:28<69:47:22,  1.26it/s]

Training Loss: 0.12097258288390003


...

Epoch: 0 --- Moving Average Loss: 0.1372260685893707:   7%|▋         | 20751/317654 [4:40:22<204:08:35,  2.48s/it] 

Training Loss: 0.1372260685893707


Epoch: 0 --- Moving Average Loss: 0.11337457998003811:   7%|▋         | 21001/317654 [4:46:24<117:02:48,  1.42s/it]

Training Loss: 0.11337457998003811


Epoch: 0 --- Moving Average Loss: 0.12701026988681405:   7%|▋         | 21140/317654 [4:52:39<228:29:34,  2.77s/it]

In [ ]:
# Mel Shape torch.Size([3, 1, 8, 32, 86])

# torch.Size([3, 8, 32, 86])
# Mel Shape torch.Size([3, 1, 128, 345])

# GENERATION

In [ ]:
def create_latent_dataset(model, dataset, batch_size, output_csv, use_sampling=False):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    dataloader = torch.utils.data.DataLoader(dataset, batch_size=batch_size, shuffle=False)

    model.to(device)
    model.eval()

    all_latents = []

    with torch.no_grad():
        for batch in dataloader:
            x_batch = batch[0].to(device)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=(device=="cuda")):
                recon_x, mu, logvar = model(x_batch)

            # Choose latent representation
            if use_sampling:
                std = torch.exp(0.5 * logvar)
                eps = torch.randn_like(std)
                z = mu + eps * std   # sampled latent
            else:
                z = mu

            # Move to CPU as float32
            z = z.detach().float().cpu()

            all_latents.append(z)

    # Concatenate all batches
    all_latents = torch.cat(all_latents, dim=0)

    # Convert to numpy
    latent_np = all_latents.numpy()

    # Save to CSV
    df = pd.DataFrame(latent_np)
    df.to_csv(output_csv, index=False)

    print(f"Saved latent dataset to {output_csv}, shape={latent_np.shape}")

model = torch.load(f"./models/GraphTransformer-Alibi-Global-Token-ALIBI-Modulation-Latent-16-Beta-0.5-Batch-2048/model.model", weights_only=False)
create_latent_dataset(model, dataset, BATCH_SIZE, "synth_latents.csv", use_sampling=False)

In [ ]:
from TransformerAutoencoder import TransformerAutoencoder

number_of_params = 225

NAME = f"PlainTransformer-Latent-{LATENT_DIM}-Beta-{BETA}-Batch-{BATCH_SIZE}-temp"
model = TransformerAutoencoder(input_size=number_of_params, latent_space=LATENT_DIM, d_model=256, depth=6, heads=4, mlp_dim=512, reparameterization=True, adjacency_matrix=None)
train_loop(model, chain=False)

## Train Models

In [ ]:
import pandas as pd
from data_processing import PatchProcessor
import data_processing

candidates = pd.read_csv("E:\\Coding\\vae-main\\generation\\dx7.candidates")
df = pd.read_csv("E:\\Coding\\vae-main\\dx7\\dx7.patchparams")

params = data_processing.get_params()
processor = PatchProcessor(params)

names = candidates["name"].tolist()
candidate_df = df.loc[(df["name"].isin(names))]
candidate_df = candidate_df.drop_duplicates(subset=["name"], keep="first")
candidate_df = candidate_df.drop(columns=candidate_df.columns[0])

candidate_df = processor.normalize(candidate_df)
df_encoded, expanded_types = processor.one_hot_dataframe(candidate_df)

mse_mask, be_mask, ce_mask, alg_mask = processor.make_masks(expanded_types)
masks = (mse_mask, be_mask, ce_mask, alg_mask)
algorithms, alibi_distances = data_processing.get_algorithms()

encoded_data_array = df_encoded.to_numpy()
x_data = torch.tensor(encoded_data_array, dtype=torch.float32)
dataset = torch.utils.data.TensorDataset(x_data)

In [ ]:
def test_performance(model, latent_size):
    BATCH_SIZE = 25
    alpha_vector = torch.tensor([0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]).unsqueeze(1).unsqueeze(1).expand(11, BATCH_SIZE, latent_size)

    candidate_tensor = torch.tensor(x_data).to("cuda")
    latent_tensors = model.encode(candidate_tensor)

    finals = []
    for index in range(len(latent_tensors)):
        item = torch.tensor(latent_tensors[index]).cpu()
        all_others = torch.tensor(latent_tensors).cpu()

        a_vector = item.unsqueeze(0).expand(11, BATCH_SIZE, -1)
        b_vectors = all_others.unsqueeze(0).expand(11, BATCH_SIZE, -1)

        interpolated_vector = (1 - alpha_vector) * a_vector
        interpolated_vector = interpolated_vector + alpha_vector * b_vectors
        finals.append(interpolated_vector)

    final_patches = []
    for final in finals:
        A, B, C = final.shape
        final = final.view(-1, LATENT_DIM).to("cuda")
        final_patch = model.generate(final)
        final = final_patch.view(A, B, -1)
        final = final.permute(1, 0, 2)
        final_patches.append(final)

    print(f"Algorithm Interpolation")

    start_index = 8
    for target_index in range(25):
        print(f"From {start_index} to {target_index}")
        for interp in range(11):
            orig_max = torch.argmax(candidate_tensor[start_index][176:176+32])
            target_max = torch.argmax(candidate_tensor[target_index][176:176+32])
            pred_max = torch.argmax(final_patches[start_index][target_index][interp][176:176+32])

            param_index = 0

            orig_param_1 = candidate_tensor[start_index][param_index]
            target_param_1 = candidate_tensor[target_index][param_index]
            pred_param_1 = final_patches[start_index][target_index][interp][param_index]

            print(f"\t\t\t{orig_max}  |  {pred_max}  |  {target_max}  |  {orig_param_1:.4f}  |  {pred_param_1:.4f}  |  {target_param_1:.4f}")